<a href="https://colab.research.google.com/github/vishvaragavanp2024aiml-lang/241501249-Deep-Learning-Lab/blob/main/EX%209.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### EX 9: TEXT GENERATION USING LSTM NETWORKS

**Objectives:**
1. Understand sequential modeling for natural language generation.
2. Train a character-level LSTM model to learn language patterns.
3. Generate text using a seed prompt and evaluate the results.
4. Analyze the fluency and creativity of LSTM-generated outputs.

#### Step 1: Load and Preprocess the Dataset
We will download the Tiny Shakespeare corpus, read the text, convert it to lowercase, and map characters to unique indices.

In [1]:
import tensorflow as tf
import numpy as np

# Download the Tiny Shakespeare dataset
text_path = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
)

# Load and clean text
text = open(text_path, 'r').read().lower()

# Create mappings from characters to indices and vice versa
chars = sorted(set(text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for i, c in enumerate(chars)}

print(f"Total unique characters: {len(chars)}")
print(f"Total dataset length: {len(text)} characters")

Total unique characters: 39
Total dataset length: 1115394 characters


#### Step 2: Create Input and Output Sequences
We'll prepare sliding window input sequences of length 40 and target output characters.

In [6]:
seq_len = 40
X = []
y = []

# Limit the corpus to the first 50,000 characters to drastically speed up model fitting
limited_text = text[:50000]

for i in range(0, len(limited_text) - seq_len, 10):
    input_seq = limited_text[i:i + seq_len]
    target_char = limited_text[i + seq_len]
    X.append([c2i[c] for c in input_seq])
    y.append(c2i[target_char])

X = np.array(X)
y = np.array(y)

print(f"Total sequences with downsampled dataset: {len(X)}")

Total sequences with downsampled dataset: 4996


#### Step 3: Build and Train the LSTM Model

In [3]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(len(chars), 64, input_length=seq_len),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(chars), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.summary()

# Fit the model (using 1 epoch for demo speed)
model.fit(X, y, batch_size=128, epochs=1)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

436/436 ━━━━━━━━━━━━━━━━━━━━ 64s 134ms/step - loss: 2.6676


#### Step 4: Define the Text Generation Function

In [4]:
def generate(seed, length=300):
    seq = [c2i[c] for c in seed.lower() if c in c2i]
    if not seq:
        # Fallback if seed characters are not in vocabulary
        seq = [0]

    for _ in range(length):
        # Keep last 'seq_len' characters
        curr_input = seq[-seq_len:]
        # Pad sequence if it is shorter than seq_len
        if len(curr_input) < seq_len:
            curr_input = [0] * (seq_len - len(curr_input)) + curr_input

        inp = np.array(curr_input).reshape(1, -1)
        pred = model.predict(inp, verbose=0)[0]

        # Select next character using predicted probability distribution
        next_idx = np.random.choice(len(pred), p=pred)
        seq.append(next_idx)

    return seed + ''.join(i2c[i] for i in seq[len(seed):])

#### Step 5: Generate and Display Text

In [5]:
print("Generated Text:\n")
print(generate("shall i compare thee to a summer's day?\n", length=300))

Generated Text:

shall i compare thee to a summer's day?

vares so'ler grileriwaned fodindant sol pusal;
fuikn tharrylc th tee loltt, a, thas co the woro e toun hankt hanf thye nhe thui is theses?
ee, munle hame: pby kom pvof
date hrarde deer out meug!, whamel ave. by ane and tot afin hoigonf, pera!n kid tothand jrewrisrarerss mepis'oola milt tho ile?
thu
